# Do-No-Harm Latent Steering: Colab runner (A100)

Push-T + frozen LeRobot Diffusion Policy + stand-in probabilistic WM + Learn-then-Test calibration.

**Code comes from GitHub ([carlo-scr/dnh-latent-steering](https://github.com/carlo-scr/dnh-latent-steering)), outputs go to Drive.**
Edit and `git push` on your laptop, then re-run the setup cell here (it pulls). Data, WMs and results live in
`MyDrive/dnh_runs`, so they survive disconnects.

Select *Runtime → Change runtime type → A100 GPU*, then run top to bottom.

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, multiprocessing; print("CPUs:", multiprocessing.cpu_count())

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
CPUs: 12


In [3]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/dnh_runs'   # all outputs (data, WMs, results)
os.makedirs(ROOT, exist_ok=True)
os.environ['ROOT'] = ROOT
os.environ['SDL_VIDEODRIVER'] = 'dummy'   # headless pygame
os.environ['MUJOCO_GL'] = 'egl'           # for the later robosuite/robomimic tasks

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Clone (first run) or pull (later runs) the code, then install. Takes ~2-3 min the first time. If pip asks to restart the runtime (torch version change), restart and re-run from the top.

In [4]:
REPO = 'https://github.com/carlo-scr/dnh-latent-steering.git'
CODE = '/content/code'
if os.path.exists(f'{CODE}/.git'):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
%cd {CODE}
!git log --oneline -1
!pip install -q -e .
!python -m pytest -q tests

cp: cannot stat '/content/drive/MyDrive/Latent_Steering/code': No such file or directory
[Errno 2] No such file or directory: '/content/code'
/content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

no tests ran in 0.00s
ERROR: file or directory not found: tests



## 0. Smoke test
Migrates the hub checkpoint into `ROOT/dp_pusht_migrated` once, checks snapshot/restore, perturbations, K-batched sampling, and paired re-runs.

In [5]:
!python scripts/00_smoke_test.py --ckpt $ROOT/dp_pusht_migrated --n 16 --K 16

python3: can't open file '/content/scripts/00_smoke_test.py': [Errno 2] No such file or directory


## 1. Collect base-policy rollouts
Nominal (success-WM data + base success rate, expect roughly 0.6-0.7 for this checkpoint) and perturbed (failure-WM data).

In [6]:
!python scripts/01_collect.py --root $ROOT --n 500 --batch 128
for p in ['friction', 'cog', 'damping', 'action_noise']:
    !python scripts/01_collect.py --root $ROOT --n 200 --batch 128 --perturbation {p} --severity 2

python3: can't open file '/content/scripts/01_collect.py': [Errno 2] No such file or directory
python3: can't open file '/content/scripts/01_collect.py': [Errno 2] No such file or directory
python3: can't open file '/content/scripts/01_collect.py': [Errno 2] No such file or directory
python3: can't open file '/content/scripts/01_collect.py': [Errno 2] No such file or directory
python3: can't open file '/content/scripts/01_collect.py': [Errno 2] No such file or directory


## 2. Train stand-in WMs (success + failure)

In [7]:
!python scripts/02_train_wm.py --root $ROOT --epochs 30

python3: can't open file '/content/scripts/02_train_wm.py': [Errno 2] No such file or directory


## 3. RQ1: week-1 GO / NO-GO
Executes every one of K candidates from each branch state. **GO if the best within-state AUROC > 0.6.**
Start small (`--n-states 64`) to measure time, then scale.

In [8]:
!python scripts/03_rq1_ranking.py --root $ROOT --n-states 64 --K 16 --batch 256
!python scripts/03_rq1_ranking.py --root $ROOT --n-states 64 --K 16 --batch 256 --perturbation friction --severity 2

python3: can't open file '/content/scripts/03_rq1_ranking.py': [Errno 2] No such file or directory
python3: can't open file '/content/scripts/03_rq1_ranking.py': [Errno 2] No such file or directory


## 4. Do-no-harm calibration (Algorithm 1)

In [9]:
!python scripts/04_calibrate_ltt.py --root $ROOT --scorer u --n-cal 200 --n-test 200 --K 16 --batch 200
!python scripts/04_calibrate_ltt.py --root $ROOT --scorer contrast --n-cal 200 --n-test 200 --K 16 --batch 200

python3: can't open file '/content/scripts/04_calibrate_ltt.py': [Errno 2] No such file or directory
python3: can't open file '/content/scripts/04_calibrate_ltt.py': [Errno 2] No such file or directory


## 5. Success under shift at the certified setting

In [10]:
!python scripts/05_eval_shift.py --root $ROOT --scorer contrast --n 100 --K 16 --batch 100

python3: can't open file '/content/scripts/05_eval_shift.py': [Errno 2] No such file or directory


In [11]:
import json, glob, pandas as pd
for f in sorted(glob.glob(f'{ROOT}/results/*.json')):
    print(f); print(json.dumps(json.load(open(f)), indent=1, default=str)[:1500]); print()